In [ ]:
import numpy as np
import psi4 
import os
# === 1. Molecule Setup ===

xyz_path = os.path.join(os.path.expanduser("~"), "DDLUCJ", "check_amplitudes", "diatomics", "NN.xyz")

with open(xyz_path, 'r') as f:
    xyz_text = f.read()

qmol = psi4.qcdb.Molecule.from_string(xyz_text, dtype='xyz')
mol = psi4.geometry(qmol.create_psi4_string_from_molecule() + "\nsymmetry c1\n")

psi4.core.clean()
psi4.core.be_quiet()

# === 2. Set Options and Run RHF ===

psi4.set_options({
    'basis': 'STO-3G',
    # 'scf_type': 'pk',
    'reference': 'rhf',
    'e_convergence': 1e-8,
    'd_convergence': 1e-8,
    'print_mos': True,
    'frozen_DOCC': [0]
})

rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
# FCIDUMP
psi4.driver.p4util.fcidump(scf_wfn)

In [ ]:
scf_wfn.nmo()

In [ ]:
import pyscf
from pyscf import ao2mo, tools
import pyscf.mcscf

# Specify molecule properties
open_shell = False
spin_sq = 0
 
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=xyz_path,
    basis="STO-3G",
    symmetry="c1",
)
 
# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())
 
# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()


# Get molecular integrals
scf = scf.from_fcidump('INTDUMP',False).run() 
# scf = pyscf.scf.RHF(mol).run()
num_orbitals = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2
cas = pyscf.mcscf.CASCI(scf, num_orbitals, (num_elec_a, num_elec_b))
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

In [ ]:
def gen_intergrals(scf_wfn,mol,active_space):
    
    num_orbitals = len(active_space)
    n_electrons = int(sum(scf_wfn.mo_occ[active_space]))
    num_elec_a = (n_electrons + mol.spin) // 2
    num_elec_b = (n_electrons - mol.spin) // 2
    cas = pyscf.mcscf.CASCI(scf_wfn, num_orbitals, (num_elec_a, num_elec_b))
    mo = cas.sort_mo(active_space, base=0)
    hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
    eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)
    e_scf = scf_wfn.e_tot
    return eri, hcore, nuclear_repulsion_energy, mo, e_scf

In [ ]:
eri_dmp, hcore_dmp, nuclear_repulsion_energy_dmp, mo_dmp, e_scf_wfn = gen_intergrals(scf.from_fcidump('INTDUMP',False).run(),mol,active_space)

In [ ]:
eri, hcore, nuclear_repulsion_energy, mo, e_scf = gen_intergrals(pyscf.scf.RHF(mol).run(),mol,active_space)

In [ ]:
abs(e_scf - e_scf_wfn)*1e3